In [ ]:
%%capture
!pip uninstall -y torchao bitsandbytes
!pip install -q torch==2.2.2 torchvision==0.17.2 torchaudio==2.2.2 --index-url https://download.pytorch.org/whl/cu118
!pip install -q transformers peft accelerate datasets trl

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Using the base model since we dropped unsloth
model_id = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)

# Load in standard 16-bit float (Supported perfectly by P100 GPU)
# We drop 4-bit quantization entirely to avoid bitsandbytes CUDA errors
model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    torch_dtype=torch.float16, 
    device_map="auto"
)

In [ ]:
from peft import LoraConfig, get_peft_model

# Critical: Enables memory saving so the 3B model fits in the P100's 16GB VRAM
model.gradient_checkpointing_enable()

config = LoraConfig(
    r=16,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, config)

In [ ]:
from datasets import load_dataset
import urllib.request

# Download your data
urllib.request.urlretrieve("https://raw.githubusercontent.com/Youmei295/DeAIze-model-source-code/main/dataset.json", "dataset.json")
dataset = load_dataset("json", data_files="dataset.json", split="train")

def formatting_func(examples):
    texts = []
    for inp, tar in zip(examples["input"], examples["target"]):
        messages = [
            {"role": "system", "content": "You are a professional Vietnamese editor. Rewrite the text to be natural, human-sounding, and concise. Remove robotic AI-isms and repetitive transitions."},
            {"role": "user", "content": inp},
            {"role": "assistant", "content": tar},
        ]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        texts.append(text)
    return {"text": texts}

dataset = dataset.map(formatting_func, batched=True)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    per_device_train_batch_size = 1, # Ultra low to fit in P100 16GB VRAM
    gradient_accumulation_steps = 8, # Compensate for small batch size
    warmup_steps = 5,
    max_steps = 60,
    learning_rate = 5e-5,
    fp16 = True, # Use standard FP16 (P100 supported)
    bf16 = False,
    logging_steps = 1,
    optim = "adamw_torch", # Standard optimizer, no bitsandbytes needed
    weight_decay = 0.01,
    lr_scheduler_type = "linear",
    seed = 3407,
    output_dir = "outputs",
    report_to = "none",
)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 2048, # Reduced sequence length to save memory
    dataset_num_proc = 2,
    packing = False,
    args = training_args,
)

trainer.train()

In [ ]:
# Save and push the LoRA adapters to the Hugging Face Hub
model.push_to_hub("Youmei295/deAIze", token=hf_token)
tokenizer.push_to_hub("Youmei295/deAIze", token=hf_token)